# Enterprise Contact Center — 04 QA Scoring Dashboard

Creates a supervisor-facing dashboard that:
- Shows overall QA score distributions and trends
- Ranks agents by performance with drill-down
- Flags outlier calls requiring human review
- Identifies coaching opportunities by criterion
- Breaks down performance by queue type

In [0]:
# Configuration - same as other notebooks
dbutils.widgets.text("catalog", "chada_demos", "Unity Catalog")
dbutils.widgets.text("schema", "contact_center_qa", "Schema")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
FQ = f"{CATALOG}.{SCHEMA}"

In [0]:
%sql
-- Overall QA metrics summary
SELECT 
  COUNT(*) AS total_calls_evaluated,
  ROUND(AVG(overall_qa_score), 2) AS avg_qa_score,
  COUNT(CASE WHEN requires_human_review THEN 1 END) AS flagged_for_review,
  ROUND(COUNT(CASE WHEN requires_human_review THEN 1 END) * 100.0 / COUNT(*), 1) AS pct_flagged,
  ROUND(AVG(greeting_score), 2) AS avg_greeting,
  ROUND(AVG(empathy_score), 2) AS avg_empathy,
  ROUND(AVG(accuracy_score), 2) AS avg_accuracy,
  ROUND(AVG(escalation_score), 2) AS avg_escalation,
  ROUND(AVG(compliance_score), 2) AS avg_compliance
FROM ${catalog}.${schema}.gold_qa_evaluations

In [0]:
%sql
-- Agent performance leaderboard
SELECT 
  agent_id,
  COUNT(*) AS calls_evaluated,
  ROUND(AVG(overall_qa_score), 2) AS avg_score,
  MIN(overall_qa_score) AS min_score,
  MAX(overall_qa_score) AS max_score,
  ROUND(AVG(greeting_score), 2) AS avg_greeting,
  ROUND(AVG(empathy_score), 2) AS avg_empathy,
  ROUND(AVG(accuracy_score), 2) AS avg_accuracy,
  ROUND(AVG(escalation_score), 2) AS avg_escalation,
  ROUND(AVG(compliance_score), 2) AS avg_compliance,
  COUNT(CASE WHEN requires_human_review THEN 1 END) AS flagged_calls
FROM ${catalog}.${schema}.gold_qa_evaluations
GROUP BY agent_id
ORDER BY avg_score DESC

In [0]:
%sql
-- Outlier calls flagged for human review
SELECT 
  call_id,
  agent_id,
  queue_type,
  overall_qa_score,
  greeting_score,
  empathy_score,
  accuracy_score,
  escalation_score,
  compliance_score,
  compliance_flags,
  coaching_notes,
  evaluated_at
FROM ${catalog}.${schema}.gold_qa_evaluations
WHERE requires_human_review = true
ORDER BY overall_qa_score ASC, evaluated_at DESC

In [0]:
%sql
-- QA scores broken down by queue/department
SELECT 
  queue_type,
  COUNT(*) AS total_calls,
  ROUND(AVG(overall_qa_score), 2) AS avg_score,
  ROUND(AVG(greeting_score), 2) AS avg_greeting,
  ROUND(AVG(empathy_score), 2) AS avg_empathy,
  ROUND(AVG(accuracy_score), 2) AS avg_accuracy,
  ROUND(AVG(escalation_score), 2) AS avg_escalation,
  ROUND(AVG(compliance_score), 2) AS avg_compliance,
  COUNT(CASE WHEN requires_human_review THEN 1 END) AS flagged_calls,
  ROUND(COUNT(CASE WHEN overall_qa_score >= 4 THEN 1 END) * 100.0 / COUNT(*), 1) AS pct_excellent
FROM ${catalog}.${schema}.gold_qa_evaluations
GROUP BY queue_type
ORDER BY avg_score DESC

In [0]:
%sql
-- Identify agents needing coaching by specific criterion
-- Shows agents whose average on any criterion falls below 3.0
WITH agent_criterion_scores AS (
  SELECT 
    agent_id,
    'Greeting & ID' AS criterion, AVG(greeting_score) AS avg_score FROM ${catalog}.${schema}.gold_qa_evaluations GROUP BY agent_id
  UNION ALL
  SELECT agent_id, 'Empathy', AVG(empathy_score) FROM ${catalog}.${schema}.gold_qa_evaluations GROUP BY agent_id
  UNION ALL
  SELECT agent_id, 'Accuracy', AVG(accuracy_score) FROM ${catalog}.${schema}.gold_qa_evaluations GROUP BY agent_id
  UNION ALL
  SELECT agent_id, 'Escalation', AVG(escalation_score) FROM ${catalog}.${schema}.gold_qa_evaluations GROUP BY agent_id
  UNION ALL
  SELECT agent_id, 'Compliance', AVG(compliance_score) FROM ${catalog}.${schema}.gold_qa_evaluations GROUP BY agent_id
)
SELECT 
  agent_id,
  criterion,
  ROUND(avg_score, 2) AS avg_criterion_score,
  CASE 
    WHEN avg_score < 2.0 THEN 'URGENT - Immediate coaching required'
    WHEN avg_score < 3.0 THEN 'WARNING - Coaching recommended'
    WHEN avg_score < 4.0 THEN 'MONITOR - Room for improvement'
    ELSE 'GOOD - Meeting expectations'
  END AS coaching_priority
FROM agent_criterion_scores
WHERE avg_score < 3.5
ORDER BY avg_score ASC

In [0]:
%sql
-- Score distribution over time (daily trend)
SELECT 
  DATE(evaluated_at) AS eval_date,
  COUNT(*) AS calls_evaluated,
  ROUND(AVG(overall_qa_score), 2) AS avg_score,
  COUNT(CASE WHEN overall_qa_score >= 4 THEN 1 END) AS excellent_calls,
  COUNT(CASE WHEN overall_qa_score < 3 THEN 1 END) AS poor_calls,
  COUNT(CASE WHEN requires_human_review THEN 1 END) AS flagged_calls
FROM ${catalog}.${schema}.gold_qa_evaluations
GROUP BY DATE(evaluated_at)
ORDER BY eval_date DESC

In [0]:
%sql
-- Customer sentiment broken down by call category
SELECT 
  call_category,
  sentiment,
  COUNT(*) AS call_count,
  ROUND(AVG(overall_qa_score), 2) AS avg_qa_score
FROM ${catalog}.${schema}.gold_qa_evaluations
GROUP BY call_category, sentiment
ORDER BY call_category, sentiment